# Notebook 03 - Modelado ML + SHAP

**MatchMind: Prediccion de Resultados de Futbol Internacional**

---

## Que hacemos en este notebook

Entrenamos cuatro modelos en escalera de complejidad y los comparamos. Esto es buena practica: nunca confies en un solo modelo. Comparar con un baseline trivial es la unica forma de saber si tu modelo realmente aprende algo.

**Los 4 modelos:**

| Modelo | Que es | Por que lo usamos |
|--------|--------|-------------------|
| Dummy | Predice siempre la clase mayoritaria | **Piso absoluto**: cualquier modelo debe superarlo |
| Logistic Regression | Modelo lineal clasico | Baseline real, muestra si el problema es lineal |
| Random Forest | 200 arboles + bagging | Modelo no lineal robusto, sin tuning |
| XGBoost | Gradient boosting | Estado del arte para datos tabulares |

Despues, aplicamos **SHAP** sobre el mejor modelo para entender:
- Que features pesan mas en general (vista global).
- Por que predice asi un partido especifico (vista local).

Al final guardamos el modelo final con `pickle` para que el notebook 04 y la demo Streamlit lo carguen sin re-entrenar.

## 0. Imports y carga del split

El split temporal ya esta guardado por el notebook 02 en `data/processed/train_test_split.pkl`. Solo cargamos.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from src.models.train import (
    fit_baseline, fit_logreg, fit_random_forest, fit_xgboost,
    save_model,
)
from src.evaluation.metrics import evaluate, comparison_table, print_report

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 200

with open('../data/processed/train_test_split.pkl', 'rb') as f:
    split = pickle.load(f)

X_train, y_train = split['X_train'], split['y_train']
X_val,   y_val   = split['X_val'],   split['y_val']
X_test,  y_test  = split['X_test'],  split['y_test']

print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')

## 1. Baseline 1: Dummy (clase mayoritaria)

El `DummyClassifier(strategy='most_frequent')` simplemente predice siempre la clase mas frecuente. En nuestro caso, siempre va a predecir `home_win`. 

**Por que importa este baseline:** si un modelo complejo no supera al Dummy por un margen significativo, es probable que tenga un bug. Es nuestro piso absoluto.

In [ ]:
dummy = fit_baseline(X_train, y_train)
y_pred_d = dummy.predict(X_test)
y_proba_d = dummy.predict_proba(X_test)
row_dummy = evaluate(y_test, y_pred_d, y_proba_d, 'Dummy')
row_dummy

**Lectura:** Accuracy 0.47 (el modelo trivial acierta al predecir 'home_win' cuando es 'home_win'). Pero F1-macro de 0.21 (terrible: no acierta empates ni victorias visitantes nunca). AUC = 0.50 (azar puro, no discrimina nada).

Esto confirma la decision del notebook 01: **Accuracy no sirve como metrica en este problema**.

## 2. Baseline 2: Logistic Regression

Modelo lineal clasico. Para que funcione bien necesita features escaladas (con media 0 y desviacion 1), asi que aplicamos `StandardScaler`. 

**Detalle critico:** el `StandardScaler` se ajusta SOLO en train (`scaler.fit(X_train)`) y se aplica a val/test (`scaler.transform(X_test)`). Si lo ajustaramos sobre todo el dataset, eso seria data leakage.

In [ ]:
logreg, scaler = fit_logreg(X_train, y_train)
X_test_s = scaler.transform(X_test)
y_pred_lr = logreg.predict(X_test_s)
y_proba_lr = logreg.predict_proba(X_test_s)
row_lr = evaluate(y_test, y_pred_lr, y_proba_lr, 'LogReg')
row_lr

**Salto enorme:** F1-macro pasa de 0.21 (Dummy) a 0.45 (LogReg). El modelo lineal ya captura muchisima informacion de las features. 

Esto sugiere que el problema **es mayormente lineal** en las features que construimos. Veremos si los modelos no lineales mejoran mucho o no.

## 3. Random Forest

Random Forest entrena 200 arboles de decision con bagging (cada arbol ve una muestra distinta de los datos). El resultado final es el promedio de las predicciones de todos los arboles.

**Por que probarlo:** los arboles capturan interacciones no lineales entre features (ej: "si el rank_diff es alto Y h2h_home_wins es alto, entonces local gana casi seguro").

In [ ]:
rf = fit_random_forest(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)
row_rf = evaluate(y_test, y_pred_rf, y_proba_rf, 'RandomForest')
row_rf

## 4. XGBoost

XGBoost es nuestro candidato a mejor modelo. A diferencia de Random Forest (que entrena arboles en paralelo), XGBoost los entrena secuencialmente: cada arbol corrige los errores del anterior.

Aqui usamos hiperparametros razonables por default. En el notebook auxiliar `run_tuning_shap.py` hacemos grid search.

In [ ]:
xgb = fit_xgboost(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
y_proba_xgb = xgb.predict_proba(X_test)
row_xgb = evaluate(y_test, y_pred_xgb, y_proba_xgb, 'XGBoost')
row_xgb

## 5. Tabla comparativa

El momento de la verdad: comparamos los 4 modelos lado a lado.

In [ ]:
table = comparison_table([row_dummy, row_lr, row_rf, row_xgb])
table

**Observacion interesante:** la diferencia en F1-macro entre LogReg (0.446), RandomForest (0.425) y XGBoost (0.456) es muy pequena. Esto confirma que el problema es mayormente lineal. La ganancia mas grande vino del feature engineering, NO del algoritmo.

**Conclusion:** XGBoost gana por un pelo, pero los tres modelos no triviales son comparables. Esto es **valioso aprender**: a veces invertir tiempo en feature engineering rinde mas que en sintonizar un algoritmo sofisticado.

## 6. Reporte detallado por clase

Las metricas agregadas (Accuracy, F1-macro) esconden lo que pasa por clase. El `classification_report` de sklearn nos da precision, recall y F1 para cada clase. Util para detectar problemas especificos.

In [ ]:
print_report(y_test, y_pred_xgb)

**Hallazgo critico:** miren el F1 de la clase `draw` (empate): solo 0.10! 

El modelo casi nunca predice empate. De los 1,360 empates reales, solo predice 85 como tal. 

**Por que pasa esto:** los empates son una clase "de frontera" - ocurren cuando dos equipos llegan justo al mismo numero de goles, lo cual es estadisticamente raro. El modelo prefiere apostar a la clase mayoritaria cuando esta indeciso.

**Esto es una limitacion conocida del problema y la documentamos honestamente en el informe.**

## 7. Confusion matrix

Visualizamos los errores. Las filas son la realidad, las columnas la prediccion. La diagonal son aciertos; lo demas son errores.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_xgb)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Away', 'Draw', 'Home'],
            yticklabels=['Away', 'Draw', 'Home'], ax=ax)
ax.set_xlabel('Prediccion')
ax.set_ylabel('Real')
ax.set_title('Confusion Matrix - XGBoost (test set)')
plt.tight_layout()
plt.savefig('../docs/figures/confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

Confirmacion visual: la fila `Draw` apenas tiene predicciones en su columna. La mayoria de empates terminan predichos como `Home`. **Subprediccion de empates** = limitacion conocida y documentada.

## 8. Explicabilidad con SHAP

Ahora viene la parte interesante: **por que** el modelo predice lo que predice.

SHAP asigna a cada feature de cada prediccion un valor que representa cuanto empujo la prediccion hacia una clase u otra. Para XGBoost usamos `TreeExplainer`, optimizado para modelos basados en arboles.

**Importante:** calcular SHAP sobre los 5,876 partidos del test seria lento. Tomamos una muestra de 500 partidos. Es suficiente para tener una vista global confiable.

In [ ]:
explainer = shap.TreeExplainer(xgb)
sample = X_test.sample(min(500, len(X_test)), random_state=42)
shap_values = explainer.shap_values(sample)

# Para XGBoost multiclase, shap_values puede ser:
# - lista de 3 arrays (API vieja)
# - array 3D (API nueva, version 3.x)
if isinstance(shap_values, list):
    print(f'SHAP: lista de {len(shap_values)} clases, cada una shape {shap_values[0].shape}')
    shap_vals_home = shap_values[2]
else:
    print(f'SHAP: array shape {shap_values.shape}')
    shap_vals_home = shap_values[:, :, 2]  # clase 2 = home_win

## 9. SHAP global - importancia de features

El barplot muestra el promedio del valor absoluto del SHAP por feature. Cuanto mas alto, mas importante es la feature en general.

In [ ]:
plt.figure(figsize=(8, 5))
shap.summary_plot(shap_vals_home, sample, plot_type='bar', show=False, plot_size=(8, 5))
plt.title('Feature importance (SHAP global) - clase Local gana')
plt.tight_layout()
plt.savefig('../docs/figures/shap_importance.png', dpi=200, bbox_inches='tight')
plt.show()

**Lectura del SHAP global:**

1. **rank_diff** (diferencia de ranking FIFA) es la feature mas influyente con un impacto promedio de 0.385.
2. **h2h_away_wins** (victorias historicas del visitante) es segundo.
3. **points_diff** y **h2h_home_wins** completan el top 4.
4. Las **rachas recientes** resultan menos importantes de lo esperado: probablemente porque el ranking FIFA ya incorpora indirectamente el desempeno reciente.

Esto confirma intuiciones del dominio: el ranking FIFA es muy informativo del resultado.

## 10. Guardar el modelo final

Persistimos el modelo XGBoost en `models/checkpoints/xgboost_model.pkl`. Esto permite que:
- El notebook 04 (LLM) lo cargue sin re-entrenar.
- La demo Streamlit lo use en produccion.
- Cualquier persona pueda reproducir nuestros resultados al clonar el repo.

In [ ]:
path = save_model(xgb, 'xgboost_model.pkl')
print(f'Modelo guardado en: {path}')
print(f'Tamano: {Path(path).stat().st_size / 1024:.1f} KB')

## Resumen del notebook

- 4 modelos entrenados: Dummy (F1=0.21) -> LogReg (0.45) -> RF (0.43) -> XGBoost (0.46).
- XGBoost gana, pero por poco. **La ganancia clave vino del feature engineering**, no del algoritmo.
- **Limitacion documentada**: el modelo subpredice empates (problema estructural del futbol).
- SHAP confirma que `rank_diff` es la feature dominante.

Para hyperparameter tuning detallado, ejecutar el script `scripts/run_tuning_shap.py` (probamos 24 combinaciones y mejoramos F1-macro de 0.456 a 0.452 en test final tuneado).

## Siguiente paso

Notebook **04_llm_integration.ipynb**: integramos Groq + Llama-3.1-8b para generar reportes narrativos a partir de los valores SHAP.